# Finite Elements — Detailed Summary

## L1: Introduction

### Model Problem — Poisson's Equation
On $\Omega=[0,1]^2$: find $u$ such that
$$-\nabla^2 u = f, \quad u(0,y)=u(1,y)=0\;(\text{Dirichlet}),\quad \frac{\partial u}{\partial y}(x,0)=\frac{\partial u}{\partial y}(x,1)=0\;(\text{Neumann}).$$
Dirichlet conditions pin the solution value on parts of the boundary, while Neumann conditions prescribe zero flux — physically, no heat flows across those edges. This mixed setup is the canonical testbed because it forces us to handle both types when deriving the weak form.

### Triangulations
**Definition.** A triangulation $\mathcal{T}$ of polygonal $\Omega\subset\mathbb{R}^2$ is a set of triangles $\{K_i\}$ s.t.:
1. $\mathrm{int}\,K_i\cap\mathrm{int}\,K_j=\emptyset$ for $i\neq j$ (no overlaps)
2. $\cup K_i=\bar{\Omega}$ (covers $\Omega$)
3. No vertex of any triangle lies in the interior of an edge of another (no hanging nodes)

The "no hanging nodes" condition is crucial: it guarantees that piecewise polynomial functions glued together across element boundaries can actually be continuous. If a node sat mid-edge on a neighbour, the values wouldn't match up and we'd lose $C^0$ continuity.

### P1 Finite Element Space
$$V_h = \{v\in C^0(\Omega): v|_{K_i}\text{ is linear }\forall K_i\in\mathcal{T}\}$$
$$\mathring{V}_h = \{v\in V_h: v(0,y)=v(1,y)=0\} \quad(\text{incorporates Dirichlet BCs})$$
A function in $V_h$ is fully determined by its vertex values — linearity on each triangle plus global continuity means there is no freedom left once you fix the nodes. The "ring" space $\mathring{V}_h$ simply zeroes out nodes on the Dirichlet boundary, baking the boundary condition directly into the space rather than enforcing it as a constraint.

### $L^2$ Norm and Space
$$\|f\|_{L^2(\Omega)}=\left(\int_\Omega|f(x)|^2\,dx\right)^{1/2}, \qquad L^2(\Omega)=\{f:\|f\|_{L^2}<\infty\}$$
Two functions identified if $\|f-g\|_{L^2}=0$. This means $L^2$ doesn't care about pointwise values — functions that differ on a set of measure zero (like a single edge) are the same function. This is exactly what lets us ignore the fact that piecewise-linear gradients are undefined on element boundaries.

### Finite Element Derivative
Defined in $L^2$: $\frac{\partial^{FE}u}{\partial x_i}\big|_K = \frac{\partial u}{\partial x_i}\big|_K$ for each $K$. Not uniquely defined on edges, but unique in $L^2$. Since edges have zero area, the ambiguity on them doesn't affect any integral, so the FE derivative is a perfectly well-defined $L^2$ function.

### Deriving the Weak Form
The core idea: we can't solve the PDE pointwise (our approximate solution isn't smooth enough to take $\nabla^2$), so we multiply by a test function and integrate to move one derivative onto the test function via integration by parts.

1. Multiply $-\nabla^2 u=f$ by test function $v\in\mathring{V}_h$, integrate over $\Omega$
2. Integration by parts on each triangle:
$$\sum_i\left(\int_{K_i}\nabla v\cdot\nabla u\,dx - \int_{\partial K_i}v\,n\cdot\nabla u\,dS\right) = \int_\Omega vf\,dx$$
3. Interior edge contributions cancel because each internal edge is shared by two triangles with opposing outward normals ($n^-=-n^+$), so the boundary integrals from adjacent elements exactly annihilate
4. Boundary terms vanish: $v=0$ on Dirichlet (we chose $v\in\mathring{V}_h$), and $n\cdot\nabla u=0$ on Neumann (that's the BC itself). Neumann conditions are called "natural" because they are automatically satisfied by the weak form without being explicitly imposed.
5. **FE approximation**: find $u_h\in\mathring{V}_h$ s.t.
$$\int_\Omega\nabla v\cdot\nabla u_h\,dx = \int_\Omega vf\,dx, \quad\forall v\in\mathring{V}_h$$

### Practical Implementation
**Nodal basis**: $\phi_i(z_j)=\delta_{ij}$ for vertices $z_j$. Each basis function is a "tent" centred at node $z_i$, equal to 1 there and decaying linearly to zero at all neighbouring nodes, vanishing everywhere else.

Expanding $u_h=\sum_j u_j\phi_j$, $v=\sum_i v_i\phi_i$ and requiring the weak form for every basis test function gives the matrix system $K\mathbf{u}=\mathbf{f}$:
$$K_{ij}=\int_\Omega\nabla\phi_i\cdot\nabla\phi_j\,dx, \qquad f_i=\int_\Omega\phi_i f\,dx$$
$K$ is sparse because $K_{ij}=0$ whenever nodes $i$ and $j$ are not connected by an edge (their basis functions have non-overlapping support). Assembly loops over elements, computing each triangle's local contribution and scattering it into the global matrix — this is efficient because each element only touches a handful of nodes.

## L2: Finite Element Spaces — Local to Global

### Ciarlet's Finite Element
**Definition.** A triple $(K,\mathcal{P},\mathcal{N})$ where:
1. $K\subset\mathbb{R}^n$: bounded closed domain (polygon) — the geometric shape of the element
2. $\mathcal{P}$: finite-dimensional space of shape functions on $K$ — e.g. all polynomials up to degree $k$
3. $\mathcal{N}=(N_0,\ldots,N_k)$: basis for dual space $\mathcal{P}'$ — the "degrees of freedom" that uniquely pin down a function in $\mathcal{P}$

The power of this abstraction is that it cleanly separates the geometry ($K$), the approximation space ($\mathcal{P}$), and the way we specify a function in that space ($\mathcal{N}$). Examples of dual functionals include point evaluation $N_i(v)=v(z_i)$, line integrals along edges, area integrals, or derivative evaluations — any linear map from $\mathcal{P}$ to $\mathbb{R}$.

### Nodal Basis
**Definition.** $\{\phi_j\}$ with $N_i(\phi_j)=\delta_{ij}$. Each basis function $\phi_j$ is the unique element of $\mathcal{P}$ whose $j$-th degree of freedom equals 1 and all others equal 0. This means the coefficients in the expansion $v=\sum_j N_j(v)\phi_j$ are simply the degrees of freedom of $v$ — no linear system to solve.

### Vandermonde Matrix
Given arbitrary basis $\{\psi_i\}$: $V_{ij}=N_j(\psi_i)$. This matrix bridges any convenient computational basis (like monomials) to the nodal basis.

**Lemma.** Nodal basis coefficients: $\phi_i=\sum_j\mu_{ij}\psi_j$ where $\mu=V^{-1}$. So constructing the nodal basis is equivalent to inverting the Vandermonde matrix.

### Unisolvence (Dual Condition Lemma)
The following are equivalent:
1. $\mathcal{N}$ is a basis for $\mathcal{P}'$
2. Vandermonde matrix $V$ is invertible
3. ($\mathcal{N}$ determines $\mathcal{P}$): $N_i(v)=0\;\forall i\Rightarrow v\equiv 0$

Unisolvence is the key well-posedness condition for a finite element: it says the degrees of freedom carry exactly enough information to reconstruct any function in $\mathcal{P}$ — no redundancy, no ambiguity. Condition 3 is the practical way to verify it: show that the only function whose DOFs all vanish is the zero function.

### 1D Lagrange Element of Degree $k$
- $K=[a,b]$, $\mathcal{P}=$ polynomials of degree $\leq k$, $N_i(v)=v(x_i)$ at $k+1$ equispaced points.
- Nodal basis: $\phi_i(x)=\prod_{j\neq i}\frac{x-x_j}{x_i-x_j}$ — classic Lagrange interpolation polynomials, each constructed to vanish at all nodes except one.
- **Corollary**: Unisolvent, since a degree-$k$ polynomial vanishing at $k+1$ points must be identically zero (fundamental theorem of algebra). This is the simplest unisolvence argument.

### Triangular Lagrange Elements (Pk)
- $K$: triangle with vertices $z_1,z_2,z_3$; $\mathcal{P}$: degree $k$ polynomials on $K$
- Nodes are placed on a regular lattice: $x_{i,j}=z_1+(z_2-z_1)\frac{i}{k}+(z_3-z_1)\frac{j}{k}$, giving $\binom{k+2}{2}$ nodes which matches $\dim(\mathcal{P})$ exactly.

**P1 unisolvence proof**: A linear polynomial $p$ vanishing at 3 non-collinear vertices must be zero. Restrict $p$ to the edge $\Pi_1$ connecting two vertices: $p|_{\Pi_1}$ is a degree-1 polynomial vanishing at 2 points, so $p=cL_1$ where $L_1$ defines $\Pi_1$. Then $p(z_1)=cL_1(z_1)=0$ forces $c=0$. The geometry of the triangle does all the work — the vertices can't be collinear.

**P2 unisolvence proof**: Here we use the 6 nodes (3 vertices + 3 edge midpoints). If $p$ vanishes at all 3 points on edge $\Pi_1$, then $p|_{\Pi_1}$ is a degree-2 polynomial with 3 roots, so $p=L_1Q_1$ by the key lemma. Similarly $Q_1$ vanishes on $\Pi_2$, giving $p=cL_1L_2$. The midpoint of the third edge then forces $c=0$.

**Key lemma for 2D**: If polynomial $p$ vanishes on the hyperplane $\Pi_L=\{x:L(x)=0\}$, then $p=LQ$ with $\deg(Q)=\deg(p)-1$. This is the 2D analogue of "if $p(a)=0$ then $(x-a)|p(x)$" and is the engine behind all triangular unisolvence proofs — each edge the polynomial vanishes on lets us factor out one linear term.

### Exotic Elements
- **Cubic Hermite**: $K$=triangle, $\mathcal{P}$=cubics (10-dimensional), $\mathcal{N}$=vertex values + vertex gradients + centre value (10 DOF). The gradient DOFs give more smoothness control at vertices than plain Lagrange.
- **Quintic Argyris**: $K$=triangle, $\mathcal{P}$=quintics (21-dimensional), $\mathcal{N}$=vertex values, gradients, Hessians + edge midpoint normal derivatives (21 DOF). This element achieves $C^1$ continuity globally, which is needed for fourth-order problems like plate bending where the weak form involves second derivatives of the solution.

### Geometric Decomposition & Global Spaces

**Continuity Lemma.** The global FE space $V$ is $C^m$ iff functions and all derivatives up to order $m$ agree on all shared mesh entities (vertices and edges). This reduces a global smoothness question to a local matching condition.

**$C^m$ geometric decomposition**: each nodal variable is assigned to a mesh entity (vertex, edge, or cell); the sub-element $(w,\mathcal{P}|_w,\mathcal{N}_w)$ restricted to each entity $w$ is itself a valid finite element. This decomposition is the mechanism for building global spaces: DOFs on shared entities (vertices, edges) are identified between neighbouring elements to enforce continuity, while DOFs on cell interiors remain independent.

- **Lagrange Pk**: has $C^0$ geometric decomposition — vertex DOFs are shared (enforcing continuity), edge DOFs are shared, cell DOFs are private. Only values, not derivatives, are matched.
- **Argyris**: has $C^1$ geometric decomposition — vertex DOFs include values, gradients, and Hessians, and edge DOFs include normal derivatives, so both the function and its first derivative are matched across elements.

**Global $C^m$ space**: Identify nodal variables on shared mesh entities between adjacent elements $\Rightarrow$ automatic $C^m$ continuity across the entire triangulation. The geometric decomposition tells you exactly which DOFs to match and guarantees the result is well-defined.

## L3: Interpolation Operators

### Local Interpolator
$$\mathcal{I}_Kv(x) = \sum_{i=0}^k N_i(v)\phi_i(x)$$
The interpolator reads off the degrees of freedom of $v$ and reconstructs the unique polynomial in $\mathcal{P}$ that matches them. It is the bridge between arbitrary functions and the finite element space.

**Properties:**
1. **Linear**: $\mathcal{I}_K(\alpha u+\beta v)=\alpha\mathcal{I}_Ku+\beta\mathcal{I}_Kv$ (follows from linearity of $N_i$)
2. **Preserves nodal values**: $N_i(\mathcal{I}_Kv)=N_i(v)$ — by construction, the interpolant has exactly the same DOFs as the original function
3. **Projection**: $\mathcal{I}_Kp=p$ for all $p\in\mathcal{P}$ — if the function is already a polynomial in $\mathcal{P}$, the interpolator reproduces it exactly (its DOFs uniquely determine it, and they haven't changed)

### Global Interpolator
$\mathcal{I}_hu|_K = \mathcal{I}_Ku$ for each $K\in\mathcal{T}_h$. We interpolate element-by-element. For Lagrange elements, the result is automatically continuous because shared vertex/edge DOFs produce the same polynomial values on both sides.

### Sobolev's Inequality (for continuous functions)
For $k>n/2$: $\|u\|_{C^\infty(\Omega)}\leq C\|u\|_{H^k(\Omega)}$. This says that if a function has enough weak derivatives in $L^2$, it must actually be continuous (and bounded). In 2D ($n=2$), we need $k\geq 2$ derivatives; in 1D, just $k\geq 1$.

For derivatives: $k-m>n/2 \Rightarrow \|u\|_{C^{m,\infty}}\leq C\|u\|_{H^k}$. This is essential because point evaluation (used in Lagrange DOFs) is only meaningful for continuous functions, and Sobolev's inequality tells us which $H^k$ spaces contain continuous functions.

### Averaged Taylor Polynomial
**Definition.** For $f\in C^{k,\infty}$, domain star-shaped w.r.t. ball $B$:
$$Q_{k,B}f(x) = \frac{1}{|B|}\int_B T^kf(y,x)\,dy$$
The idea is to average the Taylor expansion over all possible base points $y\in B$, rather than expanding around a single point. This smooths out the construction and gives better-behaved error estimates that involve only the size of the domain and Sobolev seminorms.

**Key property:** $D^\beta Q_{k,B}f = Q_{k-|\beta|,B}D^\beta f$ — differentiating the averaged Taylor polynomial is the same as taking the averaged Taylor polynomial of the derivative. This "commuting" property is what makes the error analysis clean.

### Taylor Polynomial Error Estimate
$$\|D^\beta(f-Q_{k,B}f)\|_{L^2(\Omega)}\leq C\frac{|\Omega|^{1/2}}{|B|^{1/2}}d^{k+1-|\beta|}|f|_{H^{k+1}(\Omega)}$$
The error in the Taylor approximation of order $k$ is controlled by the $(k+1)$-th seminorm — exactly the derivatives the Taylor polynomial can't capture. The factor $d^{k+1-|\beta|}$ shows higher-order Taylor polynomials give faster convergence as the domain shrinks.

### Interpolation Error Chain
The strategy is four steps: prove the bound on a reference element, extend to error, scale to arbitrary size, then sum globally.

1. **Bound on $\mathcal{I}_{K_1}$** (diameter-1 reference triangle): $\|\mathcal{I}_{K_1}u\|_{H^k}\leq C\|u\|_{H^k}$
   - *Proof*: triangle inequality on nodal basis expansion, then Sobolev inequality to bound pointwise DOFs by $H^k$ norms. This is the "interpolator is bounded" step — it doesn't blow up norms.

2. **Local error** (diameter 1): $|\mathcal{I}_{K_1}u-u|_{H^i}\leq C|u|_{H^{k+1}}$
   - *Proof*: add and subtract $Q_{k,B}u$. Since $Q_{k,B}u\in\mathcal{P}$, the projection property gives $\mathcal{I}_K Q_{k,B}u=Q_{k,B}u$, so the interpolation error is controlled by the Taylor approximation error, which is controlled by $|u|_{H^{k+1}}$. The key insight: the interpolator exactly reproduces polynomials, so its error is entirely due to the part of $u$ that polynomials can't capture.

3. **Scaling** (diameter $d$): $|\mathcal{I}_Ku-u|_{H^i(K)}\leq C_Kd^{k+1-i}|u|_{H^{k+1}(K)}$
   - *Proof*: change of variables $x\to x/d$ maps $K$ to a diameter-1 reference triangle. Each derivative picks up a factor of $1/d$, and the Jacobian contributes $d^n$ to the integral. Collecting powers of $d$ gives the scaling $d^{k+1-i}$. This is why smaller elements give better approximation — the error shrinks as a power of element diameter.

4. **Global estimate**: $\|\mathcal{I}_hu-u\|_{H^i(\Omega)}\leq Ch^{k+1-i}|u|_{H^{k+1}(\Omega)}$
   - *Proof*: sum $\|\cdot\|^2$ over all elements and use $d_K\leq h$ (mesh size). Requires a positive minimum aspect ratio (no degenerate slivers), since $C_K$ in step 3 depends on element shape. The final rate $h^{k+1-i}$ tells us: degree-$k$ elements converge at rate $k+1$ in $L^2$ and rate $k$ in $H^1$.

## L4: Finite Element Problems — Solvability & Stability

### Hilbert Space Foundations
The abstract setting for FE problems: we need a space with an inner product (to measure angles and projections) that is also complete (limits of Cauchy sequences stay in the space). This completeness is what lets variational arguments go through.

| Concept | Definition |
|---------|----------|
| Vector space | $V$ with $+$ and $\times$, closed, zero element |
| Bilinear form | $b:V\times V\to\mathbb{R}$, linear in each argument |
| Inner product | Symmetric bilinear form with $(v,v)\geq 0$ and $(v,v)=0\iff v=0$ |
| $L^2$ inner product | $(f,g)_{L^2}=\int_\Omega fg\,dx$ |
| $H^1$ inner product | $(f,g)_{H^1}=\int_\Omega fg+\nabla f\cdot\nabla g\,dx$ |
| Norm | $\|v\|=\sqrt{(v,v)}$ for inner product space |
| Complete | Every Cauchy sequence has a limit in $V$ |
| Hilbert space | Complete inner product space |

All finite-dimensional inner product spaces are complete (this is a theorem from linear algebra) $\Rightarrow$ FE spaces with $L^2$ or $H^1$ inner products are automatically Hilbert. Completeness only becomes a non-trivial issue for the infinite-dimensional continuous spaces $H^k(\Omega)$.

### Linear Functionals
- $L:H\to\mathbb{R}$ bounded iff $|L(u)|\leq C\|u\|_H$ — the functional can't grow faster than linearly in the norm
- Bounded $\iff$ continuous for linear functionals (a special fact for linear maps; nonlinear maps can be bounded but discontinuous)
- **Dual space** $H'$: space of all continuous linear functionals on $H$
- **Dual norm**: $\|L\|_{H'}=\sup_{0\neq v}\frac{L(v)}{\|v\|_H}$ — the tightest constant $C$ in the boundedness inequality
- **Riesz Representation**: $\forall L\in H'$, $\exists!$ $u\in H$ s.t. $L(v)=(u,v)$ and $\|u\|_H=\|L\|_{H'}$. This is the fundamental theorem connecting abstract functionals to concrete elements — every continuous linear functional "is" an inner product with some fixed element, and the norm is preserved. It's what ultimately lets us convert variational problems into function-finding problems.

### Linear Variational Problem
Find $u\in V$ s.t. $b(u,v)=F(v)$ $\forall v\in V$. This is the abstract form of all our FE problems: $b$ encodes the differential operator (via integration by parts) and $F$ encodes the right-hand side and boundary data.

### Continuity & Coercivity
- **Continuous**: $|b(u,v)|\leq M\|u\|_V\|v\|_V$ — the bilinear form is bounded; it can't produce arbitrarily large outputs from bounded inputs
- **Coercive**: $b(u,u)\geq\gamma\|u\|_V^2$ — feeding the same function into both slots gives a result that controls the norm squared. This is the crucial "lower bound" condition: it prevents the bilinear form from having a non-trivial null space, ensuring uniqueness.

Together, continuity and coercivity say $b$ behaves like an inner product: it's bounded above (continuity) and bounded below (coercivity) relative to $\|\cdot\|_V^2$.

### Lax-Milgram Theorem
If $b$ continuous and coercive, $F$ continuous, then $\exists!$ $u\in V$ with $\|u\|_V\leq\frac{1}{\gamma}\|F\|_{V'}$. This is the workhorse existence theorem: it guarantees a unique solution and gives a stability estimate (the solution can't be larger than $1/\gamma$ times the data). When $b$ is also symmetric, Lax-Milgram follows from Riesz representation; for non-symmetric $b$, the proof uses Banach's fixed-point theorem.

### Stability
A sequence of FE discretisations is **stable** if $\gamma_h\to\gamma>0$ and $\|F\|_{V_h'}\to c<\infty$ as $h\to 0$. Stability means the discrete problems don't degenerate as the mesh refines — the solution remains bounded independently of $h$. If $\gamma_h\to 0$, the discrete system becomes increasingly ill-conditioned.

### Helmholtz Problem
$b(u,v)=(u,v)_{H^1}=\int_\Omega(uv+\nabla u\cdot\nabla v)\,dx$. This is the simplest model problem because $b$ literally is the $H^1$ inner product.

- *Continuity*: $|b(u,v)|=|(u,v)_{H^1}|\leq\|u\|_{H^1}\|v\|_{H^1}$ by Cauchy-Schwarz, so $M=1$.
- *Coercivity*: $b(u,u)=\|u\|_{H^1}^2$, so $\gamma=1$.

Both constants are 1 and $h$-independent $\Rightarrow$ unconditionally stable. This is the ideal case against which harder problems are benchmarked.

### Poisson (Neumann) — Mean Estimate
For the Poisson problem with all-Neumann BCs, constants are in the kernel of $b(u,v)=\int\nabla u\cdot\nabla v$. We remove this degeneracy by working in the subspace $\bar{V}_h=\{u:\bar{u}=0\}$ of zero-mean functions.

The **mean estimate** $\|u-\bar{u}\|_{L^2}\leq C|u|_{H^1}$ (a Poincaré-type inequality for zero-mean functions) then gives $\|u\|_{L^2}^2\leq C^2|u|_{H^1}^2$, which implies $\|u\|_{H^1}^2=\|u\|_{L^2}^2+|u|_{H^1}^2\leq(1+C^2)|u|_{H^1}^2=(1+C^2)b(u,u)$ $\Rightarrow$ coercive on $\bar{V}_h$. The physical intuition: if you nail down the mean, the $H^1$ seminorm controls the full norm — gradients alone determine the function up to a constant.

### Poisson (Dirichlet) — Poincaré Inequality
With Dirichlet BCs on $\Gamma_0$, the function vanishes on part of the boundary, so we can control $\|u\|_{L^2}$ by $|u|_{H^1}$ using the Poincaré inequality. The proof chain is: FE divergence theorem $\to$ trace theorem ($\|u\|_{L^2(\partial\Omega)}\leq C\|u\|_{H^1}$, bounding boundary values by interior norms) $\to$ Poincaré inequality for functions vanishing on $\Gamma_0$. The trace theorem is what connects boundary conditions to interior behaviour.

## L5: Convergence of Finite Element Approximations

### Weak Derivatives
**Definition.** $D_w^\alpha f\in L^1_{loc}$ defined by:
$$\int_\Omega\phi\,D_w^\alpha f\,dx = (-1)^{|\alpha|}\int_\Omega D^\alpha\phi\,f\,dx, \quad\forall\phi\in C_0^\infty(\Omega)$$
The weak derivative is defined by what integration by parts *would* give if $f$ were smooth — we transfer all derivatives onto a smooth test function $\phi$ with compact support (so boundary terms vanish). A function "has a weak derivative" if there exists an $L^1_{loc}$ function satisfying this identity. This lets us differentiate functions that aren't classically differentiable.

**Key equivalences:**
- For $u\in C^0$ FE space: FE derivative $=$ weak derivative (the piecewise-defined FE derivative satisfies the integration-by-parts identity because interior edge terms cancel, just as in the weak form derivation)
- For $u\in C^{|\alpha|}$: strong derivative $=$ weak derivative (integration by parts works classically)
- For discontinuous FE: weak $\neq$ FE derivative in general (jumps across elements produce Dirac-delta contributions in the weak derivative that the piecewise FE derivative misses)

### Sobolev Spaces
$$H^k(\Omega)=\{u\in L^1_{loc}: \|u\|_{H^k}<\infty\}$$
$H^k$ collects all functions with $k$ weak derivatives in $L^2$. It is a Hilbert space (complete under $\|\cdot\|_{H^k}$). The inclusion $H^l\subset H^k$ for $l>k$ is strict: more derivatives in $L^2$ is a stronger condition.

**$H=W$ theorem**: $H^k\cap C^\infty$ is dense in $H^k$ — any $H^k$ function can be approximated by smooth functions. This is technically important because it lets us prove results for smooth functions (where classical calculus works) and extend by density.

**Sobolev's inequality** (general): $k>n/2$ $\Rightarrow$ $\|u\|_{L^\infty}\leq C\|u\|_{H^k}$, and there is a continuous representative. The critical threshold $k>n/2$ reflects a dimensional trade-off: in higher dimensions, functions need more weak derivatives to be forced into continuity.

### Strong $\leftrightarrow$ Weak Equivalence (Helmholtz)
This establishes that the variational form and the PDE are truly equivalent (not just one-directional), provided the solution is regular enough.

- **Strong $\Rightarrow$ Weak**: If $u\in H^2$ solves $u-\nabla^2u=f$ pointwise, multiply by test function $v$ and integrate by parts — this is just the derivation of the weak form, running forward.
- **Weak $\Rightarrow$ Strong**: If $u\in H^2$ solves the variational form, we recover the PDE:
  1. Test with $v\in C_0^\infty$ (which vanish on the boundary): the variational equation becomes $\int(u-\nabla^2u-f)v=0$ for all such $v$, and since $C_0^\infty$ is dense in $L^2$, this forces $u-\nabla^2u=f$ a.e. in $\Omega$
  2. For the boundary condition: substitute the PDE back into the variational form to get $\int_{\partial\Omega}\frac{\partial u}{\partial n}v\,dS=0$ for arbitrary $v$, which forces the Neumann condition $\frac{\partial u}{\partial n}=0$

### Galerkin Approximation
For $V_h\subset V=H^k$: find $u_h\in V_h$ s.t. $b(u_h,v)=F(v)$ $\forall v\in V_h$. The Galerkin method simply restricts the variational problem to the finite-dimensional subspace $V_h$. Because $V_h\subset V$, the continuity and coercivity of $b$ on $V$ are automatically inherited on $V_h$ (the constants can only improve, never worsen) $\Rightarrow$ Lax-Milgram applies directly.

### Céa's Lemma
$$\|u-u_h\|_V\leq\frac{M}{\gamma}\min_{v\in V_h}\|u-v\|_V$$

This is the central result connecting approximation theory to FE convergence: the FE solution $u_h$ is quasi-optimal — it's within a factor $M/\gamma$ of the best possible approximation in $V_h$. The ratio $M/\gamma$ measures how far $b$ is from being an inner product; for Helmholtz, $M/\gamma=1$ so $u_h$ *is* the best approximation.

**Proof sketch:**
1. **Galerkin orthogonality**: $b(u-u_h,v)=0$ $\forall v\in V_h$ — the error is $b$-orthogonal to the discrete space. This is the FE analogue of orthogonal projection.
2. For any $v\in V_h$: $\gamma\|u-u_h\|^2\leq b(u-u_h,u-u_h)=b(u-u_h,u-v)+\underbrace{b(u-u_h,v-u_h)}_{=0\text{ (Galerkin orth.)}}\leq M\|u-u_h\|\|u-v\|$
3. Divide by $\|u-u_h\|$, minimise over $v$. The coercivity lower-bounds the error, continuity upper-bounds the cross term, and Galerkin orthogonality eliminates the $v-u_h$ piece.

### Interpolation Error in $H^k$ (extended from L3)
All L3 results extend from smooth functions to $H^{k+1}$ functions: replace $C^{l,\infty}$ norms by $W^l_\infty$ norms, prove for smooth functions, then pass to the limit using the $H=W$ density theorem. This is why $H=W$ matters — it lets the interpolation theory developed for smooth functions apply to the broader Sobolev setting.

### Convergence of Helmholtz
Combine Céa's lemma with the interpolation estimate by choosing $v=\mathcal{I}_hu$:
$$\|u_h-u\|_{H^1}\leq\frac{M}{\gamma}\|u-\mathcal{I}_hu\|_{H^1}\leq Ch^k|u|_{H^{k+1}}$$
For degree-$k$ elements, the $H^1$ error converges at rate $h^k$. Higher polynomial degree buys faster convergence, but requires the exact solution to have correspondingly more regularity ($u\in H^{k+1}$).

### Aubin-Nitsche Trick ($L^2$ estimates)
The $H^1$ estimate above is the best Céa's lemma can give directly, but the $L^2$ error should be better (we lose nothing to derivatives). The Aubin-Nitsche trick recovers this extra order by introducing a dual (adjoint) problem.

**Elliptic regularity**: On a convex domain, $|w|_{H^2}\leq C\|f\|_{L^2}$ for the solution of $w-\nabla^2w=f$. This is the crucial regularity assumption — it says the PDE "gains two derivatives," so $L^2$ data produces $H^2$ solutions. It can fail on non-convex domains (where corner singularities limit regularity).

**$L^2$ estimate**: $\|u_h-u\|_{L^2}\leq Ch^{k+1}\|u\|_{H^{k+1}}$

**Proof sketch** (the "duality argument"):
1. Solve the dual problem: $w-\nabla^2w=u-u_h$ (the error itself is the right-hand side)
2. Then $\|u-u_h\|_{L^2}^2=(u-u_h,u-u_h)_{L^2}=b(w,u-u_h)=b(w-\mathcal{I}_hw,u-u_h)$ where the last step uses Galerkin orthogonality to subtract off $\mathcal{I}_hw\in V_h$
3. Apply continuity: $\leq M\|w-\mathcal{I}_hw\|_{H^1}\cdot\|u-u_h\|_{H^1}\leq Ch\cdot|w|_{H^2}\cdot\|u-u_h\|_{H^1}$ using the interpolation estimate on $w$
4. Elliptic regularity gives $|w|_{H^2}\leq C\|u-u_h\|_{L^2}$; substitute and divide both sides by $\|u-u_h\|_{L^2}$

The net effect: each application of Galerkin orthogonality + interpolation gains one power of $h$, so $L^2$ error is one order better than $H^1$ error. For P1 elements: $O(h)$ in $H^1$, $O(h^2)$ in $L^2$.

## L6: Stokes Equation (Mastery)

### Strong Form
$$-2\mu\nabla\cdot\varepsilon(u)+\nabla p=f, \quad \nabla\cdot u=0, \quad \varepsilon(u)=\tfrac{1}{2}(\nabla u+\nabla u^T)$$
- $u$: velocity (vector), $p$: pressure (scalar), $\mu$: viscosity, $f$: body force
- Boundary condition: $u=0$ on $\partial\Omega$ (no-slip)
- Pressure unique up to constant $\Rightarrow$ impose $\int_\Omega p=0$

This models slow viscous flow (Stokes regime). The incompressibility constraint $\nabla\cdot u=0$ couples velocity and pressure, and the pressure acts as a Lagrange multiplier enforcing this constraint. Because pressure has no time derivative or diffusion term, it behaves fundamentally differently from velocity — this asymmetry is the source of all the difficulties.

### Function Spaces
$$V=(\mathring{H}^1(\Omega))^n, \quad Q=\mathring{L}^2(\Omega)=\{p\in L^2:\int p=0\}$$
Velocity lives in $H^1$ (one derivative for the strain tensor, plus zero boundary conditions), while pressure only needs to be in $L^2$ — it appears undifferentiated in the weak form. The zero-mean condition on $Q$ removes the constant-pressure ambiguity.

### Variational Form
Find $(u,p)\in V\times Q$ s.t.
$$a(u,v)+b(v,p)=\int_\Omega f\cdot v, \quad b(u,q)=0, \quad \forall(v,q)\in V\times Q$$
where $a(u,v)=2\mu\int_\Omega\varepsilon(u):\varepsilon(v)$, $b(v,q)=-\int_\Omega q\,\nabla\cdot v$.

The first equation is the momentum balance (tested against velocity), and the second is the incompressibility constraint (tested against pressure). This is a **saddle-point problem**: we're simultaneously minimising over $u$ and maximising over $p$ — the Lagrange multiplier structure.

### Why Lax-Milgram Fails
The combined form $c(U,W)=a(u,v)+b(v,p)+b(u,q)$ is **not coercive**: $c(U,U)=a(u,u)+2b(u,p)$, and taking $u=0$ gives $c((0,p),(0,p))=0$ for any $p\neq 0$. The pressure doesn't appear quadratically anywhere — there is no "pressure diffusion" — so coercivity over the full product space $V\times Q$ is impossible. This is the fundamental reason Stokes requires new theory beyond Lax-Milgram.

### Inf-Sup Condition
$$\inf_{0\neq q\in Q}\sup_{0\neq v\in V}\frac{b(v,q)}{\|v\|_V\|q\|_Q}\geq\beta>0$$

This replaces the missing coercivity for the pressure. It says: for every pressure $q$, there exists a velocity $v$ that "detects" it through the bilinear form $b$ — no pressure is invisible to the velocity space. Equivalently, $\|B^*q\|_{V'}\geq\beta\|q\|_Q$ for all $q$ (the operator $B^*$ mapping pressure to velocity functionals is injective with a uniform lower bound).

In finite dimensions, $B^*$ injective $\iff$ $B$ surjective (rank-nullity theorem). Surjectivity of $B$ means every divergence-free constraint can actually be enforced — there are "enough" velocity DOFs relative to pressure DOFs.

### Brezzi's Theorem
Define kernel $Z=\{u\in V:b(u,q)=0\;\forall q\}$ — the divergence-free velocities. Assume:
1. $a$ coercive on $Z$: $a(u,u)\geq\alpha\|u\|_V^2$ $\forall u\in Z$ — we only need coercivity on the constraint manifold, not all of $V$
2. Inf-sup for $b$ with constant $\beta>0$ — this controls the pressure

Then $\exists!$ $(u,p)$ with stability bounds:
$$\|u\|_V\leq\frac{1}{\alpha}\|F\|_{V'}+\frac{2M}{\alpha\beta}\|G\|_{Q'}$$
$$\|p\|_Q\leq\frac{2M}{\alpha\beta}\|F\|_{V'}+\frac{2M^2}{\alpha\beta^2}\|G\|_{Q'}$$
The velocity bound comes from coercivity on $Z$; the pressure bound comes from the inf-sup condition. Notice the pressure bound scales as $1/\beta^2$ — a weak inf-sup constant means poor pressure stability, which is exactly what happens with bad element choices.

### Discrete Inf-Sup
For FE spaces $V_h\subset V$, $Q_h\subset Q$, need:
$$\inf_{q\in Q_h}\sup_{v\in V_h}\frac{b(v,q)}{\|v\|_V\|q\|_Q}\geq\beta_h>0$$
with $\beta_h$ bounded away from zero independently of $h$. **This is not automatic from the continuous inf-sup!** Restricting to subspaces can destroy the inf-sup condition — the "detecting" velocity for a given pressure may not lie in $V_h$. This is where the choice of element pair becomes critical.

### Element Pairs

| Pair | $V_h$ | $Q_h$ | Inf-sup? |
|------|-------|-------|----------|
| P1-P1 | $(P1)^d$ | P1 | **NO** ($\beta_h\to 0$) |
| MINI | $(P1+B3)^d$ | P1 | Yes |
| Taylor-Hood | $(P2)^d$ | P1 | Yes |

**P1-P1 fails** because there are too many pressure DOFs relative to velocity DOFs — certain non-constant pressure modes produce zero divergence when tested against all P1 velocities, creating "spurious pressure modes." The MINI element fixes this by enriching the velocity space with cubic bubble functions (one per triangle, vanishing on all edges) that can detect these problematic pressure modes. Taylor-Hood uses quadratic velocities with linear pressures, which provides enough velocity resolution naturally.

### Fortin's Trick
If there exists a bounded projection $\Pi_h:V\to V_h$ satisfying $b(v-\Pi_hv,q_h)=0$ for all $v\in V$ and $q_h\in Q_h$, then discrete inf-sup holds with $\beta_h=\beta/C_\Pi$ where $C_\Pi=\|\Pi_h\|$.

The idea: to verify discrete inf-sup, we need to find a discrete velocity $v_h\in V_h$ that detects a given $q_h$ just as well as the continuous velocity $v\in V$ does. The Fortin operator $\Pi_h$ manufactures such a $v_h=\Pi_hv$ by projecting while preserving the divergence pairing. The condition $b(v-\Pi_hv,q_h)=0$ ensures no information is lost in the projection.

### Error Estimate for Mixed Problems
$$\|u_h-u\|_V\leq\frac{4MM_b}{\alpha\beta_h}E_u+\frac{M_b}{\alpha}E_p$$
$$\|p_h-p\|_Q\leq\frac{3M^2M_b}{\alpha\beta_h^2}E_u+\frac{3MM_b}{\alpha\beta_h}E_p$$
where $E_u=\inf_{u_I\in V_h}\|u-u_I\|_V$, $E_p=\inf_{p_I\in Q_h}\|p-p_I\|_Q$.

This is the mixed analogue of Céa's lemma: the FE error is bounded by best-approximation errors in each variable, but the constants involve both $\alpha$ (coercivity on the kernel) and $\beta_h$ (inf-sup). The pressure error depends on $1/\beta_h^2$, so a deteriorating inf-sup constant disproportionately damages pressure accuracy. Combined with interpolation estimates, Taylor-Hood gives $O(h^2)$ velocity error in $H^1$ and $O(h)$ pressure error in $L^2$.